In [76]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [77]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://doi.org/10.1038/s41586-022-04518-2"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41586-022-04518-2/MediaObjects/41586_2022_4518_MOESM4_ESM.xlsx"

# don't include in header
table_name = "41586_2022_4518_MOESM4_ESM.xlsx"
table_name = "clean_degs.xlsx" # local name

## Read in file

In [78]:
excel = pd.read_excel(table_name, sheet_name = ["all markers", "immune markers"], skiprows = 1)

df = pd.concat(excel.values(), ignore_index = True)

df = df.rename(columns={"cluster": "cell_type_label"})

In [79]:
df.head()

,Unnamed: 0,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cell_type_label,gene
0,CXCL14,0.0,3.699825,0.426,0.082,0.0,ASPC,CXCL14
1,NEGR1,0.0,3.668332,0.867,0.247,0.0,ASPC,NEGR1
2,DCN,0.0,3.565394,0.960,0.506,0.0,ASPC,DCN
3,LAMA2,0.0,3.417902,0.829,0.266,0.0,ASPC,LAMA2
4,APOD,0.0,3.383653,0.493,0.120,0.0,ASPC,APOD


## Unfiltered

In [ ]:
unfiltered_df = df.copy()
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df.drop(['Unnamed: 0', 'p_val', 'avg_log2FC', 'pct.1', 'pct.2', 'p_val_adj'], axis=1, inplace=True) 
save = unfiltered_df.to_dict(orient = "records")
save

## Save Unfiltered

In [65]:
with open("evidence_unfiltered.json", "w") as f:
    json.dump(save, f, indent = 4) 

## Perform the filtering

In [80]:
col_pval = "p_val_adj"
col_lfc = "avg_log2FC"
col_gene = "gene"
col_ct = "cell_type_label"
col_rank = "pct.1"

In [81]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [82]:
markers

,Unnamed: 0,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cell_type_label,gene
16890,VNN2,5.344361e-103,2.283059,0.205,0.012,1.685237e-98,neutrophil,VNN2
31650,IL1R13,3.182986e-17,2.241460,0.212,0.058,1.003691e-12,hNeu,IL1R1
16926,JAML4,1.844625e-42,2.327117,0.274,0.046,5.816657e-38,neutrophil,JAML
15121,KCNQ54,0.000000e+00,2.205825,0.282,0.032,0.000000e+00,nk_cell,KCNQ5
16909,ITGAX5,1.309428e-60,2.431961,0.295,0.039,4.129021e-56,neutrophil,ITGAX
...,...,...,...,...,...,...,...,...
3035,SORBS1,0.000000e+00,3.347645,0.967,0.170,0.000000e+00,adipocyte,SORBS1
3033,PDE3B,0.000000e+00,3.599566,0.974,0.218,0.000000e+00,adipocyte,PDE3B
30880,RABGAP1L5,2.575032e-20,2.237582,0.974,0.639,8.119849e-16,hpDC,RABGAP1L
3045,GHR,0.000000e+00,2.701779,0.977,0.417,0.000000e+00,adipocyte,GHR


In [83]:
markers[col_ct].value_counts()

cell_type_label
adipocyte         20
pericyte          20
hpDC              20
LEC               20
hPlasmablast      20
mesothelium       20
macrophage        20
endothelial       20
ASPC              20
SMC               20
endometrium       20
neutrophil        19
hASDC             18
hMac2             10
t_cell            10
mast_cell         10
hcDC1              9
nk_cell            8
hMast              8
monocyte           7
b_cell             5
hTreg              5
hMac1              5
hTcell2            3
hNeu               3
dendritic_cell     2
hMono2             2
hMac3              1
hBcell             1
hNK                1
Name: count, dtype: int64

In [84]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cell_type_label
hNeu              0.351667
neutrophil        0.465684
nk_cell           0.507375
t_cell            0.532300
hMast             0.535625
b_cell            0.538400
hTcell2           0.545000
mast_cell         0.546200
hMac3             0.564000
hPlasmablast      0.575700
hTreg             0.576000
hNK               0.583000
pericyte          0.587250
monocyte          0.598714
hcDC1             0.613333
SMC               0.624800
dendritic_cell    0.648000
hMac2             0.649300
endometrium       0.659750
hASDC             0.686500
ASPC              0.738400
hMac1             0.742000
endothelial       0.743850
LEC               0.748200
macrophage        0.749200
hpDC              0.788500
mesothelium       0.807750
hBcell            0.819000
hMono2            0.882500
adipocyte         0.926800
Name: pct.1, dtype: float64

## Find Ensembl ID

In [71]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [72]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [73]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json

In [90]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = None


save = df.to_dict(orient = "records")

In [91]:
save

[{'cell_type_label': 'neutrophil',
  'gene': 'VNN2',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'hNeu',
  'gene': 'IL1R1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'neutrophil',
  'gene': 'JAML',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'nk_cell',
  'gene': 'KCNQ5',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'neutrophil',
  'gene': 'ITGAX',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'neutrophil',
  'gene': 'RASSF2',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': None},
 {'cell_type_label': 'neutrophil',
  'gene': 'BASP1',
  'organism': 'homo_sapiens',
  

## Save evidence

In [92]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 